<a href="https://colab.research.google.com/github/malakbayramovaa/Machine-Learning/blob/main/cow%2C_goat_diseases.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import zipfile

zip_path = "/content/drive/MyDrive/agrolab_split.zip"
extract_path = "/content/"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [ ]:
data_dir = "/content/agrolab_split"

In [ ]:
import os

print(os.listdir("/content"))

['.config', 'Augmented', 'drive', 'sample_data']


In [ ]:
import os
import shutil
import random

random.seed(42)

# ✅ SƏNİN DATASET BURADADIR
SOURCE_DIR = "/content/Augmented"
BASE_DIR = "/content/agrolab_split"

train_ratio = 0.7
val_ratio = 0.2

# köhnə varsa sil
if os.path.exists(BASE_DIR):
    shutil.rmtree(BASE_DIR)

for class_name in os.listdir(SOURCE_DIR):
    class_path = os.path.join(SOURCE_DIR, class_name)

    if not os.path.isdir(class_path):
        continue

    images = os.listdir(class_path)
    random.shuffle(images)

    train_end = int(len(images) * train_ratio)
    val_end = int(len(images) * (train_ratio + val_ratio))

    train_imgs = images[:train_end]
    val_imgs = images[train_end:val_end]
    test_imgs = images[val_end:]

    for split, img_list in zip(
        ["train", "validation", "test"],
        [train_imgs, val_imgs, test_imgs]
    ):
        split_path = os.path.join(BASE_DIR, split, class_name)
        os.makedirs(split_path, exist_ok=True)

        for img in img_list:
            src = os.path.join(class_path, img)
            dst = os.path.join(split_path, img)
            shutil.copy(src, dst)

print("✅ Dataset bölündü!")

✅ Dataset bölündü!


In [ ]:
print(os.listdir("/content/agrolab_split"))

['validation', 'train', 'test']


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    "/content/agrolab_split/train",
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical'
)

Found 4266 images belonging to 11 classes.


In [ ]:
val_data = train_gen.flow_from_directory(
    "/content/agrolab_split/validation",
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical'
)

Found 1218 images belonging to 11 classes.


In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

# freeze (çox vacib)
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dense(11, activation='softmax')
])

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 11)             │         1,419 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,423,371 (9.24 MB)

 Trainable params: 165,387 (646.04 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
print("Model hazırdır")

Model hazırdır


In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

Epoch 1/10
134/134 ━━━━━━━━━━━━━━━━━━━━ 57s 286ms/step - accuracy: 0.5143 - loss: 1.4190 - val_accuracy: 0.6223 - val_loss: 1.0771
Epoch 2/10
134/134 ━━━━━━━━━━━━━━━━━━━━ 8s 60ms/step - accuracy: 0.7365 - loss: 0.7806 - val_accuracy: 0.6987 - val_loss: 0.8817
Epoch 3/10
134/134 ━━━━━━━━━━━━━━━━━━━━ 8s 62ms/step - accuracy: 0.8094 - loss: 0.5868 - val_accuracy: 0.7266 - val_loss: 0.7982
Epoch 4/10
134/134 ━━━━━━━━━━━━━━━━━━━━ 8s 58ms/step - accuracy: 0.8497 - loss: 0.4536 - val_accuracy: 0.7184 - val_loss: 0.8058
Epoch 5/10
134/134 ━━━━━━━━━━━━━━━━━━━━ 8s 59ms/step - accuracy: 0.8936 - loss: 0.3487 - val_accuracy: 0.7397 - val_loss: 0.7346
Epoch 6/10
134/134 ━━━━━━━━━━━━━━━━━━━━ 8s 57ms/step - accuracy: 0.9283 - loss: 0.2623 - val_accuracy: 0.7529 - val_loss: 0.7342
Epoch 7/10
134/134 ━━━━━━━━━━━━━━━━━━━━ 8s 56ms/step - accuracy: 0.9470 - loss: 0.2084 - val_accuracy: 0.7652 - val_loss: 0.7418
Epoch 8/10
134/134 ━━━━━━━━━━━━━━━━━━━━ 9s 64ms/step - accuracy: 0.9655 - loss: 0.1527 - val_ac

In [ ]:
# model.save("agrolab_model.h5")

In [ ]:
import os
print(os.listdir("/content"))

['.config', 'Augmented', 'drive', 'agrolab_split', 'sample_data']


In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),   # 🔥 YENİ (çox vacib)
    layers.Dense(11, activation='softmax')
])

In [ ]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 11)             │         1,419 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,423,371 (9.24 MB)

 Trainable params: 165,387 (646.04 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)


Epoch 1/10
134/134 ━━━━━━━━━━━━━━━━━━━━ 31s 156ms/step - accuracy: 0.3608 - loss: 1.8522 - val_accuracy: 0.6117 - val_loss: 1.2239
Epoch 2/10
134/134 ━━━━━━━━━━━━━━━━━━━━ 8s 63ms/step - accuracy: 0.5441 - loss: 1.3025 - val_accuracy: 0.6691 - val_loss: 1.0426
Epoch 3/10
134/134 ━━━━━━━━━━━━━━━━━━━━ 7s 54ms/step - accuracy: 0.6104 - loss: 1.1107 - val_accuracy: 0.6757 - val_loss: 0.9293
Epoch 4/10
134/134 ━━━━━━━━━━━━━━━━━━━━ 8s 61ms/step - accuracy: 0.6564 - loss: 0.9735 - val_accuracy: 0.7102 - val_loss: 0.8664
Epoch 5/10
134/134 ━━━━━━━━━━━━━━━━━━━━ 7s 53ms/step - accuracy: 0.6955 - loss: 0.8748 - val_accuracy: 0.7036 - val_loss: 0.8547
Epoch 6/10
134/134 ━━━━━━━━━━━━━━━━━━━━ 8s 60ms/step - accuracy: 0.7225 - loss: 0.7870 - val_accuracy: 0.6962 - val_loss: 0.8233
Epoch 7/10
134/134 ━━━━━━━━━━━━━━━━━━━━ 8s 60ms/step - accuracy: 0.7496 - loss: 0.7076 - val_accuracy: 0.7422 - val_loss: 0.7479
Epoch 8/10
134/134 ━━━━━━━━━━━━━━━━━━━━ 9s 53ms/step - accuracy: 0.7726 - loss: 0.6370 - val_ac

In [ ]:
# modeli save et
model.save("/content/agrolab_savedmodel.keras")


In [ ]:
# training bitəndən sonra birbaşa save et
model.save("/content/agrolab_savedmodel.keras")

from google.colab import files
files.download("/content/agrolab_savedmodel.keras")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip install tensorflow==2.12.0

ERROR: Could not find a version that satisfies the requirement tensorflow==2.12.0 (from versions: 2.16.0rc0, 2.16.1, 2.16.2, 2.17.0rc0, 2.17.0rc1, 2.17.0, 2.17.1, 2.18.0rc0, 2.18.0rc1, 2.18.0rc2, 2.18.0, 2.18.1, 2.19.0rc0, 2.19.0, 2.19.1, 2.20.0rc0, 2.20.0, 2.21.0rc0, 2.21.0rc1, 2.21.0)
ERROR: No matching distribution found for tensorflow==2.12.0


In [ ]:
model.export("/content/agrolab_savedmodel")

!zip -r /content/agrolab_savedmodel.zip /content/agrolab_savedmodel

from google.colab import files
files.download("/content/agrolab_savedmodel.zip")

Saved artifact at '/content/agrolab_savedmodel'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_481')
Output Type:
  TensorSpec(shape=(None, 11), dtype=tf.float32, name=None)
Captures:
  136519468521936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136519468523088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136519468528656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136519468522512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136519468526928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136519468520400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136519468526736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136519468527312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136519468521744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136519468527120: TensorSpec(shape=(), dtype=tf.resource, name=None)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(11, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

model.save("/content/agrolab_final_model.keras")

Epoch 1/10
134/134 ━━━━━━━━━━━━━━━━━━━━ 32s 169ms/step - accuracy: 0.3455 - loss: 1.9139 - val_accuracy: 0.5805 - val_loss: 1.3163
Epoch 2/10
134/134 ━━━━━━━━━━━━━━━━━━━━ 22s 60ms/step - accuracy: 0.5309 - loss: 1.3502 - val_accuracy: 0.6371 - val_loss: 1.0601
Epoch 3/10
134/134 ━━━━━━━━━━━━━━━━━━━━ 8s 58ms/step - accuracy: 0.6055 - loss: 1.1203 - val_accuracy: 0.7011 - val_loss: 0.9275
Epoch 4/10
134/134 ━━━━━━━━━━━━━━━━━━━━ 8s 61ms/step - accuracy: 0.6648 - loss: 0.9768 - val_accuracy: 0.6864 - val_loss: 0.8766
Epoch 5/10
134/134 ━━━━━━━━━━━━━━━━━━━━ 7s 53ms/step - accuracy: 0.6962 - loss: 0.8671 - val_accuracy: 0.7299 - val_loss: 0.8002
Epoch 6/10
134/134 ━━━━━━━━━━━━━━━━━━━━ 8s 62ms/step - accuracy: 0.7344 - loss: 0.7749 - val_accuracy: 0.7323 - val_loss: 0.7843
Epoch 7/10
134/134 ━━━━━━━━━━━━━━━━━━━━ 8s 57ms/step - accuracy: 0.7576 - loss: 0.7111 - val_accuracy: 0.7356 - val_loss: 0.7390
Epoch 8/10
134/134 ━━━━━━━━━━━━━━━━━━━━ 8s 62ms/step - accuracy: 0.7625 - loss: 0.6497 - val_a